# 01 — Data Understanding & Initial Exploration

## 1. Project Objective
The goal of this project is **Disease Diagnosis Prediction Using Clinical Data and Classification Models**.
We aim to analyze clinical attributes from heart disease patients to build reliable predictive models that classify whether a patient has heart disease (binary target).

In this notebook, we perform initial data loading, schema inspection, data quality verification (missing values and duplicates), original target (`num`) analysis, and binary target creation (`target = (num > 0)`).


## 2. Import Libraries
We import standard data science libraries (`pandas`, `numpy`) and our project modules (`src.config`, `src.data_loader`).


In [1]:
import sys
from pathlib import Path

# Locate project root containing src/
root_dir = Path.cwd()
for p in [root_dir] + list(root_dir.parents):
    if (p / 'src' / 'config.py').exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

import pandas as pd
import numpy as np

from src.config import DATASET_PATH, RAW_COLUMNS, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, BOOLEAN_FEATURES
from src.data_loader import load_raw_data, create_binary_target, load_model_data


## 3. Load Dataset
Load the raw dataset from `data/raw/heart_disease.csv` using the reusable `load_raw_data()` function.


In [2]:
df_raw = load_raw_data()
print("Raw dataset loaded successfully from:", DATASET_PATH)


Raw dataset loaded successfully from: C:\projects\Disease-Diagnosis-Prediction\data\raw\heart_disease.csv


## 4. Dataset Shape
Inspect total rows and total columns in the raw dataset.


In [3]:
rows, cols = df_raw.shape
print(f"Dataset Shape: {rows} rows x {cols} columns")


Dataset Shape: 918 rows x 11 columns


## 5. First 10 Rows
Display the first 10 patient records.


In [4]:
df_raw.head(10)


,age,sex,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,num
0,63,Male,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,0
1,67,Male,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,2
2,67,Male,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,1
3,37,Male,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,0
4,41,Female,typical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,0
5,56,Male,typical angina,120.0,236.0,False,normal,178.0,False,0.8,0
6,62,Female,asymptomatic,140.0,268.0,False,lv hypertrophy,160.0,False,3.6,3
7,57,Female,asymptomatic,120.0,354.0,False,normal,163.0,True,0.6,0
8,63,Male,asymptomatic,130.0,254.0,False,lv hypertrophy,147.0,False,1.4,2
9,53,Male,asymptomatic,140.0,203.0,True,lv hypertrophy,155.0,True,3.1,1


## 6. Last 5 Rows
Display the last 5 patient records.


In [5]:
df_raw.tail(5)


,age,sex,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,num
913,54,Female,asymptomatic,127.0,333.0,True,st-t abnormality,154.0,False,0.0,1
914,62,Male,typical angina,130.0,139.0,False,st-t abnormality,140.0,False,0.5,0
915,55,Male,asymptomatic,122.0,223.0,True,st-t abnormality,100.0,False,0.0,2
916,58,Male,asymptomatic,130.0,385.0,True,lv hypertrophy,140.0,False,0.5,0
917,62,Male,typical angina,120.0,254.0,False,lv hypertrophy,93.0,True,0.0,1


## 7. Column Names
Inspect column headers in the raw dataset.


In [6]:
print("Column Names:")
for i, col in enumerate(df_raw.columns, 1):
    print(f"  {i}. {col}")


Column Names:
  1. age
  2. sex
  3. cp
  4. trestbps
  5. chol
  6. fbs
  7. restecg
  8. thalch
  9. exang
  10. oldpeak
  11. num


## 8. Data Types
Check pandas data types for each feature.


In [7]:
df_raw.dtypes


age           int64
sex          object
cp           object
trestbps    float64
chol        float64
fbs            bool
restecg      object
thalch      float64
exang          bool
oldpeak     float64
num           int64
dtype: object

## 9. Missing-Value Analysis
Check for missing (null/NaN) values across all columns.


In [8]:
missing_summary = pd.DataFrame({
    'Missing_Count': df_raw.isna().sum(),
    'Missing_Percentage': (df_raw.isna().mean() * 100).round(2)
})
missing_summary


,Missing_Count,Missing_Percentage
age,0,0.0
sex,0,0.0
cp,0,0.0
trestbps,0,0.0
chol,0,0.0
fbs,0,0.0
restecg,0,0.0
thalch,0,0.0
exang,0,0.0
oldpeak,0,0.0


## 10. Duplicate Analysis
Identify complete duplicate rows in the raw dataset.


In [9]:
duplicate_count = df_raw.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")


Number of duplicate rows: 0


## 11. Numerical Descriptive Statistics
Compute summary statistics (mean, std, min, median, max) for numerical features.


In [10]:
df_raw[NUMERICAL_FEATURES].describe().T


,count,mean,std,min,25%,50%,75%,max
age,918.0,53.510893,9.432617,28.0,47.00,54.0,60.00,77.0
trestbps,918.0,132.141612,17.924706,80.0,120.00,130.0,140.00,200.0
chol,918.0,199.862745,109.154522,0.0,177.25,223.0,267.00,603.0
thalch,918.0,137.689542,25.153455,60.0,120.00,140.0,155.75,202.0
oldpeak,918.0,0.855120,1.058450,-2.6,0.00,0.5,1.50,6.2


## 12. Unique Values for Categorical/Boolean Columns
Inspect distinct categorical and boolean feature values.


In [11]:
for col in CATEGORICAL_FEATURES + BOOLEAN_FEATURES:
    unique_vals = df_raw[col].unique()
    print(f"Column '{col}' ({len(unique_vals)} unique values): {unique_vals}")


Column 'sex' (2 unique values): ['Male' 'Female']
Column 'cp' (3 unique values): ['typical angina' 'asymptomatic' 'non-anginal']
Column 'restecg' (3 unique values): ['lv hypertrophy' 'normal' 'st-t abnormality']
Column 'fbs' (2 unique values): [ True False]
Column 'exang' (2 unique values): [False  True]


## 13. Original `num` Distribution
Examine the multiclass integer distribution of original target `num` (0 to 4).


In [12]:
num_counts = df_raw['num'].value_counts().sort_index()
num_pct = df_raw['num'].value_counts(normalize=True).sort_index() * 100

pd.DataFrame({
    'Diagnosis Stage (num)': num_counts.index,
    'Count': num_counts.values,
    'Percentage (%)': num_pct.round(2).values
})


,Diagnosis Stage (num),Count,Percentage (%)
0,0,410,44.66
1,1,265,28.87
2,2,108,11.76
3,3,107,11.66
4,4,28,3.05


## 14. Binary Target Creation
Create the binary target variable where:
- `0` = No Heart Disease (`num == 0`)
- `1` = Heart Disease (`num > 0`)

The original `num` column is preserved in the DataFrame.


In [13]:
df_with_target = create_binary_target(df_raw)
df_with_target[['num', 'target']].head(10)


,num,target
0,0,0
1,2,1
2,1,1
3,0,0
4,0,0
5,0,0
6,3,1
7,0,0
8,2,1
9,1,1


## 15. Binary Target Distribution
Analyze the class balance of the binary target variable `target`.


In [14]:
target_counts = df_with_target['target'].value_counts()
target_pct = df_with_target['target'].value_counts(normalize=True) * 100

pd.DataFrame({
    'Class': ['Heart Disease (1)', 'No Heart Disease (0)'],
    'Count': [target_counts.get(1, 0), target_counts.get(0, 0)],
    'Percentage (%)': [target_pct.get(1, 0.0), target_pct.get(0, 0.0)]
}).round(2)


,Class,Count,Percentage (%)
0,Heart Disease (1),508,55.34
1,No Heart Disease (0),410,44.66


## 16. Feature/Target Separation
Separate feature matrix `X` and target vector `y` using `load_model_data()`.
Verify that `num` is excluded from feature matrix `X`.


In [15]:
X, y = load_model_data()
print(f"Feature Matrix X Shape: {X.shape}")
print(f"Target Vector y Shape: {y.shape}")
print(f"'num' in X columns? {'num' in X.columns}")
print(f"'target' in X columns? {'target' in X.columns}")
print("\nFeatures in X:", list(X.columns))


Feature Matrix X Shape: (918, 10)
Target Vector y Shape: (918,)
'num' in X columns? False
'target' in X columns? False

Features in X: ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'sex', 'cp', 'restecg', 'fbs', 'exang']


## 17. Data-Quality Findings
Key data quality findings from inspection:
- **Completeness**: 0 missing values across all 918 rows and 11 columns.
- **Uniqueness**: 0 duplicate rows detected.
- **Column Naming**: `thalch` is used instead of standard `thalach`.
- **Target Distribution**: Moderate balance — 55.34% (508 cases) positive for heart disease (`target=1`), 44.66% (410 cases) negative (`target=0`).
- **Feature Isolation**: `num` is correctly isolated from feature matrix `X` to prevent target leakage.


## 18. Summary
In Phase 1 data understanding:
1. Validated the raw dataset of 918 patient records and 11 features.
2. Verified zero missing values and zero duplicate rows.
3. Created the binary classification target `target` from `num`.
4. Verified feature matrix `X` containing 10 input features (excluding `num`).
5. Configured production preprocessor pipeline supporting numerical scaling, categorical encoding, and boolean casting.
